1. IMPORTACION DE LIBRERIAS Y CARGA DEL DATASET

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

# Configuración de visualización de Pandas para poder leer mejor textos largos.
pd.set_option("display.max_colwidth", 180)

df = pd.read_csv("c:/Users/Usuario/Desktop/proyecto-final-data-analysis/data/steam_games_reviews.csv")

df.head(10)

,app_id,name,reviews
0,876320,Hyperspace Delivery Service,"“...this looks like the kind of thing that would have warranted a massive cardboard box, at least three 3.5″ floppy disks and a chunky manual dense with evocative line-art and ..."
1,876500,HyperDot,"“Easy to learn, but maddeningly cerebrum-busting to master. I survived 13.2 seconds — even after multiple tries. But I didn’t want to stop indulging.” Washington Post “You pick..."
2,876840,999,“A game about literally throwing your head off every surface in a room until you reach the exit. Excellent!” 85 – GameHype
3,876850,WhiteLily 1：丽丽公主,“这款游戏在玩法上属于角色扮演，在游戏性上以剧情为主，同时包含一些解谜元素和少量的回合制战斗。作为一款剧情向游戏，主线节奏安排得十分紧凑，跌宕起伏，并且刻意弱化了解谜和战斗元素：谜题虽有一定难度，但在卡关时会进行提示，并且只有剧情怪会触发战斗，也没有迷宫。游戏虽然由RPGMAKER开发，但是全部重制了UI，并加入了完善的任务指引系统，立绘十分精美，背...
4,877040,Samurai Wish,“I thought this is another trash game. Instead of it I was enthralled of Samurai Wish. Simple visuals but enjoyable and very hardcore gameplay. Stylish music is different for e...
5,877300,Liquid Sunshine,"“Its narration in the form of comics, in a rather dark and violent universe, is very reminiscent of the universe of Sin City.” 7.8/10 – Xbox World France “It’s a brutal tale of..."
6,877810,Anodyne 2: Return to Dust,"“A one-of-a-kind splicing of PS1 with 16-bit aesthetics and formal conventions, streaked with self-aware humour, sorrow and yearning” Eurogamer “Has a dreamy tone and big, surr..."
7,878330,Whip! Whip!,"“Played every coin-op in the annals of arcade history? Whip! Whip! offers something new but delightfully retro, mixing mechanics of Bubble Bobble and Bionic Commando.” 79/100 –..."
8,878440,Fantasy Ball,"“We have had a fantastic amount of fun exploring all that the game has to offer. Three generations of my family played, as couch potato games are a hobby of ours. Myself, two o..."
9,878580,GARAGE: Bad Trip,"“Not meant to be a quick spray and pray shooter, the variety in the challenges you’ll face, the line of sight mechanic, and the plain old weirdness throughout make it worthwhil..."


2. ANALISIS INICIAL
- cantidad de filas y columnas
- tipos de datos
- valores nulos
- filas duplicadas
- cantidad de juegos distintos

In [2]:
print("Dimensiones:", df.shape)

print("\nColumnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())

print("\nFilas duplicadas:", df.duplicated().sum())
print("Juegos distintos por nombre:", df["name"].nunique())
print("app_id distintos:", df["app_id"].nunique())

Dimensiones: (13082, 3)

Columnas:
['app_id', 'name', 'reviews']

Tipos de datos:
app_id     int64
name         str
reviews      str
dtype: object

Valores nulos:
app_id     0
name       0
reviews    0
dtype: int64

Filas duplicadas: 0
Juegos distintos por nombre: 13036
app_id distintos: 13082


## 3. Longitud de los textos originales

- `longitud_caracteres`: cantidad de caracteres del texto.
- `cantidad_palabras`: cantidad aproximada de palabras separadas por espacios.

count: cantidad total de valores/registros analizados.

mean: promedio de todos los valores.

std: desviación estándar. Indica cuánto varían los valores respecto al promedio.

min: valor más pequeño encontrado.

25%: primer cuartil. El 25% de los registros tiene un valor igual o menor a este.

50%: mediana. El 50% de los registros está por debajo de este valor y el otro 50% por encima.

75%: tercer cuartil. El 75% de los registros tiene un valor igual o menor a este.

max: valor más grande encontrado.

In [3]:
# Cantidad total de caracteres de cada celda de reviews.
df["longitud_caracteres"] = df["reviews"].astype(str).str.len()

# Cantidad aproximada de palabras de cada celda de reviews.
df["cantidad_palabras"] = df["reviews"].astype(str).str.split().str.len()

# Resumen estadístico.
df[["longitud_caracteres", "cantidad_palabras"]].describe()

,longitud_caracteres,cantidad_palabras
count,13082.000000,13082.000000
mean,324.527824,54.337104
std,240.523020,41.303625
min,2.000000,1.000000
25%,160.000000,26.000000
50%,282.000000,47.000000
75%,437.000000,73.000000
max,2913.000000,531.000000


## 5. reseñas/citas concatenadas

    Extrae los fragmentos encerrados entre comillas.
    Devuelve una lista.

In [4]:
def extraer_citas(texto):

    texto = str(texto)
    citas = re.findall(r'“([^”]+)”', texto)

    #espacios de mas y citas vacías.
    return [cita.strip() for cita in citas if cita.strip()]


# Guardamos las citas encontradas en una column
df["citas_extraidas"] = df["reviews"].apply(extraer_citas)

#  cantidad de citas en cada registro.
df["cantidad_citas"] = df["citas_extraidas"].str.len()

print("Cantidad de citas detectadas por registro:")
print(df["cantidad_citas"].value_counts().sort_index())

Cantidad de citas detectadas por registro:
cantidad_citas
0     1034
1     2637
2     2329
3     7038
4       27
5        7
6        8
7        1
10       1
Name: count, dtype: int64


| Citas encontradas | Registros |                                        |
| ----------------: | --------: | -------------------------------------------------------- |
|                 0 |     1.034 | 1.034 registros donde no se detectaron citas entre `“ ”` |
|                 1 |     2.637 | 2.637 registros contienen una cita                       |
|                 2 |     2.329 | 2.329 registros contienen dos citas                      |
|                 3 |     7.038 | 7.038 registros contienen tres citas                     |
|                 4 |        27 | 27 registros contienen cuatro citas                      |
|                 5 |         7 | 7 registros contienen cinco citas                        |
|                 6 |         8 | 8 registros contienen seis citas                         |
|                 7 |         1 | 1 registro contiene siete citas                          |
|                10 |         1 | 1 registro contiene diez citas                           |

más de la mitad del dataset contiene tres textos entre comillas en una misma celda

De los 13.082 registros: 9.411 (≈71,94 %) contienen dos o más citas.

### Registros considerados concatenados

Consideramos como **registro concatenado** aquel que contiene más de una cita detectada.

In [5]:
resenas_concatenadas = df[df["cantidad_citas"] > 1].copy()

cantidad_concatenadas = len(resenas_concatenadas)
porcentaje_concatenadas = cantidad_concatenadas / len(df) * 100

print("Registros originales:", len(df))
print("Registros con más de una cita:", cantidad_concatenadas)
print(f"Porcentaje con citas concatenadas: {porcentaje_concatenadas:.2f}%")

resenas_concatenadas[["name", "reviews", "cantidad_citas"]].head(10)

Registros originales: 13082
Registros con más de una cita: 9411
Porcentaje con citas concatenadas: 71.94%


,name,reviews,cantidad_citas
0,Hyperspace Delivery Service,"“...this looks like the kind of thing that would have warranted a massive cardboard box, at least three 3.5″ floppy disks and a chunky manual dense with evocative line-art and ...",3
1,HyperDot,"“Easy to learn, but maddeningly cerebrum-busting to master. I survived 13.2 seconds — even after multiple tries. But I didn’t want to stop indulging.” Washington Post “You pick...",3
3,WhiteLily 1：丽丽公主,“这款游戏在玩法上属于角色扮演，在游戏性上以剧情为主，同时包含一些解谜元素和少量的回合制战斗。作为一款剧情向游戏，主线节奏安排得十分紧凑，跌宕起伏，并且刻意弱化了解谜和战斗元素：谜题虽有一定难度，但在卡关时会进行提示，并且只有剧情怪会触发战斗，也没有迷宫。游戏虽然由RPGMAKER开发，但是全部重制了UI，并加入了完善的任务指引系统，立绘十分精美，背...,3
4,Samurai Wish,“I thought this is another trash game. Instead of it I was enthralled of Samurai Wish. Simple visuals but enjoyable and very hardcore gameplay. Stylish music is different for e...,3
5,Liquid Sunshine,"“Its narration in the form of comics, in a rather dark and violent universe, is very reminiscent of the universe of Sin City.” 7.8/10 – Xbox World France “It’s a brutal tale of...",3
6,Anodyne 2: Return to Dust,"“A one-of-a-kind splicing of PS1 with 16-bit aesthetics and formal conventions, streaked with self-aware humour, sorrow and yearning” Eurogamer “Has a dreamy tone and big, surr...",3
9,GARAGE: Bad Trip,"“Not meant to be a quick spray and pray shooter, the variety in the challenges you’ll face, the line of sight mechanic, and the plain old weirdness throughout make it worthwhil...",3
11,BORIS the Mutant Bear with a Gun,“Get a Job!” 9000 – My Mom “You can compare this game to classic DOOM! And it becomes exciting!” Not Refund! –,2
13,KIN,"“House of Secrets’ platformer puzzle game, KIN, had me swearing like a salty dog for all the right reasons.” 9/10 – VR the Gamers “Games of KIN’s quality and few and far betwee...",2
15,Lawless Lands,“This extremely indie game by Corrosive Studios is a gem worth taking a closer look at.” Fcfreepresspa “I'm having the best time with it. It's a real joy to play.” Steam Review...,3


## 6. Separación de las reseñas en filas independientes

In [6]:
def preparar_resenas(texto):
    """
    Devuelve una lista de reseñas individuales.

    - Si encuentra citas entre “ ”, devuelve cada cita por separado.
    - Si no encuentra citas, devuelve el texto original dentro de una lista.
    """
    texto = str(texto).strip()
    citas = extraer_citas(texto)

    if citas:
        return citas

    if texto:
        return [texto]

    return []


# Trabajamos sobre una copia para no modificar el DataFrame original.
df_preparado = df.copy()

# Guardamos el texto completo original para trazabilidad.
df_preparado["texto_original"] = df_preparado["reviews"]

# Cada celda pasa a contener una lista de una o varias reseñas.
df_preparado["resenas_individuales"] = df_preparado["reviews"].apply(preparar_resenas)

# explode() convierte cada elemento de la lista en una fila independiente.
df_reviews = df_preparado[
    ["app_id", "name", "texto_original", "cantidad_citas", "resenas_individuales"]
].explode("resenas_individuales", ignore_index=True)

# Renombramos la nueva columna para trabajar con un nombre más simple.
df_reviews = df_reviews.rename(columns={"resenas_individuales": "review"})

# Eliminamos filas vacías que pudieran aparecer y limpiamos espacios.
df_reviews = df_reviews.dropna(subset=["review"]).copy()
df_reviews["review"] = df_reviews["review"].astype(str).str.strip()
df_reviews = df_reviews[df_reviews["review"] != ""].copy()

print("Registros antes de separar:", len(df))
print("Reseñas/documentos después de separar:", len(df_reviews))

df_reviews.head(10)

Registros antes de separar: 13082
Reseñas/documentos después de separar: 29651


,app_id,name,texto_original,cantidad_citas,review
0,876320,Hyperspace Delivery Service,"“...this looks like the kind of thing that would have warranted a massive cardboard box, at least three 3.5″ floppy disks and a chunky manual dense with evocative line-art and ...",3,"...this looks like the kind of thing that would have warranted a massive cardboard box, at least three 3.5″ floppy disks and a chunky manual dense with evocative line-art and l..."
1,876320,Hyperspace Delivery Service,"“...this looks like the kind of thing that would have warranted a massive cardboard box, at least three 3.5″ floppy disks and a chunky manual dense with evocative line-art and ...",3,These are the voyages of the good ship Rock Paper Spacegun. Our one year mission; to deliver a parcel.
2,876320,Hyperspace Delivery Service,"“...this looks like the kind of thing that would have warranted a massive cardboard box, at least three 3.5″ floppy disks and a chunky manual dense with evocative line-art and ...",3,"...a charming, retro-inspired experience. Pixel graphics and chill synth tunes create a solid package for a game that wants to bring back a familiar experience ... it seeks to ..."
3,876500,HyperDot,"“Easy to learn, but maddeningly cerebrum-busting to master. I survived 13.2 seconds — even after multiple tries. But I didn’t want to stop indulging.” Washington Post “You pick...",3,"Easy to learn, but maddeningly cerebrum-busting to master. I survived 13.2 seconds — even after multiple tries. But I didn’t want to stop indulging."
4,876500,HyperDot,"“Easy to learn, but maddeningly cerebrum-busting to master. I survived 13.2 seconds — even after multiple tries. But I didn’t want to stop indulging.” Washington Post “You pick...",3,"You pick up, you play, you die, you keep playing."
5,876500,HyperDot,"“Easy to learn, but maddeningly cerebrum-busting to master. I survived 13.2 seconds — even after multiple tries. But I didn’t want to stop indulging.” Washington Post “You pick...",3,This one will be great for fast-thinking puzzle fiends.
6,876840,999,“A game about literally throwing your head off every surface in a room until you reach the exit. Excellent!” 85 – GameHype,1,A game about literally throwing your head off every surface in a room until you reach the exit. Excellent!
7,876850,WhiteLily 1：丽丽公主,“这款游戏在玩法上属于角色扮演，在游戏性上以剧情为主，同时包含一些解谜元素和少量的回合制战斗。作为一款剧情向游戏，主线节奏安排得十分紧凑，跌宕起伏，并且刻意弱化了解谜和战斗元素：谜题虽有一定难度，但在卡关时会进行提示，并且只有剧情怪会触发战斗，也没有迷宫。游戏虽然由RPGMAKER开发，但是全部重制了UI，并加入了完善的任务指引系统，立绘十分精美，背...,3,这款游戏在玩法上属于角色扮演，在游戏性上以剧情为主，同时包含一些解谜元素和少量的回合制战斗。作为一款剧情向游戏，主线节奏安排得十分紧凑，跌宕起伏，并且刻意弱化了解谜和战斗元素：谜题虽有一定难度，但在卡关时会进行提示，并且只有剧情怪会触发战斗，也没有迷宫。游戏虽然由RPGMAKER开发，但是全部重制了UI，并加入了完善的任务指引系统，立绘十分精美，背景...
8,876850,WhiteLily 1：丽丽公主,“这款游戏在玩法上属于角色扮演，在游戏性上以剧情为主，同时包含一些解谜元素和少量的回合制战斗。作为一款剧情向游戏，主线节奏安排得十分紧凑，跌宕起伏，并且刻意弱化了解谜和战斗元素：谜题虽有一定难度，但在卡关时会进行提示，并且只有剧情怪会触发战斗，也没有迷宫。游戏虽然由RPGMAKER开发，但是全部重制了UI，并加入了完善的任务指引系统，立绘十分精美，背...,3,作为一款RPG游戏，比起战斗，该作更注重剧情跟解谜（内还含一些比较有趣的小游戏），其中部分场景带有微恐怖元素，作者也很贴心地将剧情模式与恐怖模式单独分离出来供玩家自由选择，而且全程都有任务提示，基本不用担心卡关。剧情发展跌宕起伏，音乐唯美动听，加上独特温馨的系统风格，让人有一口气玩下去的冲动，另外后续免费更新的章节内容对于低售价来说更是良心。喜爱温馨...
9,876850,WhiteLily 1：丽丽公主,“这款游戏在玩法上属于角色扮演，在游戏性上以剧情为主，同时包含一些解谜元素和少量的回合制战斗。作为一款剧情向游戏，主线节奏安排得十分紧凑，跌宕起伏，并且刻意弱化了解谜和战斗元素：谜题虽有一定难度，但在卡关时会进行提示，并且只有剧情怪会触发战斗，也没有迷宫。游戏虽然由RPGMAKER开发，但是全部重制了UI，并加入了完善的任务指引系统，立绘十分精美，背...,3,Gal与RPG玩法混合在一起的游戏，游戏全程快节奏，充满着解谜向与兽耳娘妹子们的剧情，玩家不会感受到无聊，游戏4元的价格也是相当亲民的。 游戏的亮点是这猫猫的立绘，与Nekopara风格有所相似，立绘我觉得可以吹爆。剧情是选择性的，不能我全都要。


Luego de separar las celdas concatenadas en reseñas individuales, terminamos con un volumen mucho mayor de reseñas: 29651

También descubrimos que los idiomas de las reseñas varían mucho. Para este proyecto solo queremos trabajar con reseñas en español e inglés.

### Identificado y limpieza de reseñas en otras lenguas



In [7]:
from langdetect import detect, DetectorFactory

# Hace que langdetect produzca resultados reproducibles.
DetectorFactory.seed = 0

def detectar_idioma(texto):
    texto = str(texto).strip()

    if(len(texto) < 3):
        return "desconocido"

    try:
        return detect(texto)

    except:
        return "desconocido"

# Detectamos el idioma de cada reseña individual.
df_reviews["idioma"] = df_reviews["review"].apply(detectar_idioma)


In [8]:
print("Cantidad de reseñas por idioma:")

print(
    df_reviews["idioma"]
    .value_counts()
)

Cantidad de reseñas por idioma:
idioma
en             27442
desconocido      667
zh-cn            163
de               138
fr               134
sl                85
it                81
ja                75
es                74
ca                71
ro                69
af                64
no                55
ru                52
nl                48
tl                46
pt                41
ko                40
so                37
et                35
da                35
cy                29
id                24
pl                19
sw                17
vi                16
tr                13
hu                10
sv                10
cs                 9
fi                 7
sq                 7
hr                 6
bg                 6
lt                 6
zh-tw              5
ar                 5
sk                 3
uk                 3
th                 1
el                 1
he                 1
lv                 1
Name: count, dtype: int64


In [9]:
# Guardamos la cantidad antes del filtrado.
total_antes = len(df_reviews)

# Eliminamos reseñas que no esten en ingles
df_reviews = df_reviews[df_reviews["idioma"]=="en"].copy()

In [10]:
total_despues = len(df_reviews)
descartadas = total_antes - total_despues
print("\n--- Resultado del filtrado ---")
print("Reseñas antes del filtrado:", total_antes)
print("Reseñas en inglés:", total_despues)
print("Reseñas descartadas:", descartadas)

print(
    "Porcentaje conservado:",
    round(total_despues / total_antes * 100, 2),
    "%"
)


--- Resultado del filtrado ---
Reseñas antes del filtrado: 29651
Reseñas en inglés: 27442
Reseñas descartadas: 2209
Porcentaje conservado: 92.55 %


### Control de calidad
-Verificamos la integridad de los datos luego de separar las reseñas concatenadas y filtrar las reseñas por su idioma.

Se revisan:
- valores nulos;
- textos vacíos;
- duplicados exactos dentro del mismo juego;
- longitud de las reseñas;
- una muestra aleatoria para inspección manual.

In [11]:
# Valores nulos y textos vacíos.
print("Reseñas nulas:", df_reviews["review"].isnull().sum())
print("Reseñas vacías:", df_reviews["review"].str.strip().eq("").sum())

# Duplicados exactos dentro del mismo juego.
duplicados = df_reviews.duplicated(subset=["app_id", "review"]).sum()
print("Duplicados exactos dentro del mismo juego:", duplicados)

# Eliminamos duplicados exactos para evitar contar dos veces el mismo texto.
df_reviews = df_reviews.drop_duplicates(subset=["app_id", "review"]).copy()
df_reviews = df_reviews.reset_index(drop=True)

# Medidas simples de longitud después de todas las transformaciones.
df_reviews["longitud_caracteres"] = df_reviews["review"].str.len()
df_reviews["cantidad_palabras"] = df_reviews["review"].str.split().str.len()

print("\nCantidad final de reseñas para NLP:", len(df_reviews))

df_reviews[["longitud_caracteres", "cantidad_palabras"]].describe()

Reseñas nulas: 0
Reseñas vacías: 0
Duplicados exactos dentro del mismo juego: 7

Cantidad final de reseñas para NLP: 27435


,longitud_caracteres,cantidad_palabras
count,27435.000000,27435.000000
mean,129.032076,22.238309
std,104.680318,18.256160
min,5.000000,1.000000
25%,62.000000,11.000000
50%,103.000000,18.000000
75%,164.000000,28.000000
max,2754.000000,468.000000


In [12]:
# Muestra aleatoria para revisar manualmente el resultado final.
df_reviews[["app_id", "name", "review", "idioma"]].sample(
    min(15, len(df_reviews)),
    random_state=42
)

,app_id,name,review,idioma
14836,700600,Evil Genius 2: World Domination,Evil Genius 2 is probably one of the best base builders I have ever played,en
6149,1561340,Berserk Boy,"Berserk Boy a ton of fun, it’s honestly one of the best new platformers I’ve played in a very long time.",en
25980,31900,Nancy Drew®: The Creature of Kapu Cave,'Nancy Drew: The Creature of Kapu Cave is aimed at gamers who like sleuthing in exotic surroundings with a lively cast of suspects. Also aimed at the many fans of the Hardy Boy...,en
273,905340,Heave Ho,Absolutely essential. It really is one of the best local co-op games available.,en
1122,999220,Amnesia: Rebirth,One of the most thrilling survival horror games in recent memory,en
4871,1395520,The Séance of Blake Manor,"If you’re a fan of narrative puzzle games, you’re going to come away with a very strong Game of the Year contender after playing",en
6049,1546910,Sophistry - Love & Despair,"Next level of quality in VN games - beautiful mix of traditionally japanese anime with K-drama and korean manwha hand painted art style. 6 endings, dynamical branching, and a q...",en
24781,261700,Eryi's Action,This game has single handedly made me rethink my whole gaming life,en
5006,1416130,Perfect Vermin,"But either way it was interesting. It's not really like. Well its actually kind of fun. Smashing stuff not hunting down the vermin, just smashing stuff in general is pretty fun...",en
20067,406730,"1,000 Heads Among the Trees","It's all pleasantly dizzying, like a fever dream.",en


### TOKENIZACION

La tokenización transforma cada reseña en una lista de palabras.

In [13]:
def tokenizar_simple(texto):
    """Convierte una reseña en una lista simple de tokens en minúsculas."""
    texto = str(texto).lower()

    # Extrae palabras en inglés y contracciones simples.
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", texto)

    return tokens

# Aplicamos la función a cada reseña.
df_reviews["tokens"] = df_reviews["review"].apply(tokenizar_simple)

# Observamos algunos ejemplos.
df_reviews[["review", "tokens"]].head(10)

,review,tokens
0,"...this looks like the kind of thing that would have warranted a massive cardboard box, at least three 3.5″ floppy disks and a chunky manual dense with evocative line-art and l...","[this, looks, like, the, kind, of, thing, that, would, have, warranted, a, massive, cardboard, box, at, least, three, floppy, disks, and, a, chunky, manual, dense, with, evocat..."
1,These are the voyages of the good ship Rock Paper Spacegun. Our one year mission; to deliver a parcel.,"[these, are, the, voyages, of, the, good, ship, rock, paper, spacegun, our, one, year, mission, to, deliver, a, parcel]"
2,"...a charming, retro-inspired experience. Pixel graphics and chill synth tunes create a solid package for a game that wants to bring back a familiar experience ... it seeks to ...","[a, charming, retro, inspired, experience, pixel, graphics, and, chill, synth, tunes, create, a, solid, package, for, a, game, that, wants, to, bring, back, a, familiar, experi..."
3,"Easy to learn, but maddeningly cerebrum-busting to master. I survived 13.2 seconds — even after multiple tries. But I didn’t want to stop indulging.","[easy, to, learn, but, maddeningly, cerebrum, busting, to, master, i, survived, seconds, even, after, multiple, tries, but, i, didn, t, want, to, stop, indulging]"
4,This one will be great for fast-thinking puzzle fiends.,"[this, one, will, be, great, for, fast, thinking, puzzle, fiends]"
5,A game about literally throwing your head off every surface in a room until you reach the exit. Excellent!,"[a, game, about, literally, throwing, your, head, off, every, surface, in, a, room, until, you, reach, the, exit, excellent]"
6,I thought this is another trash game. Instead of it I was enthralled of Samurai Wish. Simple visuals but enjoyable and very hardcore gameplay. Stylish music is different for ea...,"[i, thought, this, is, another, trash, game, instead, of, it, i, was, enthralled, of, samurai, wish, simple, visuals, but, enjoyable, and, very, hardcore, gameplay, stylish, mu..."
7,"Slay hundreds of enemies with your Hattori Hanzo, make Uma Thurman jealous. Here comes a good top-down action game where you get to be a bloodthirsty samurai!","[slay, hundreds, of, enemies, with, your, hattori, hanzo, make, uma, thurman, jealous, here, comes, a, good, top, down, action, game, where, you, get, to, be, a, bloodthirsty, ..."
8,"Samurai Wish is a classic topdown action game. Kill waves of enemies using your katana or throw some deadly shurikens. Great music selection (from ambiant to hiphop style), fas...","[samurai, wish, is, a, classic, topdown, action, game, kill, waves, of, enemies, using, your, katana, or, throw, some, deadly, shurikens, great, music, selection, from, ambiant..."
9,"Its narration in the form of comics, in a rather dark and violent universe, is very reminiscent of the universe of Sin City.","[its, narration, in, the, form, of, comics, in, a, rather, dark, and, violent, universe, is, very, reminiscent, of, the, universe, of, sin, city]"


### STOPWORDS Y LIMPIEZA DE TOKENS

utilizamos `ENGLISH_STOP_WORDS` de Scikit-learn.

Se conservan las negaciones `no`, `not` y `never`, porque pueden cambiar de forma importante el significado de una opinión.


In [14]:
stopwords = set(ENGLISH_STOP_WORDS)

# Conservamos negaciones importantes para análisis de opiniones.
for palabra in ["no", "not", "never"]:
    stopwords.discard(palabra)

def limpiar_tokens(tokens):
    #Elimina stopwords y tokens de una sola letra
    return [
        token
        for token in tokens
        if token not in stopwords and len(token) > 1
    ]

df_reviews["tokens_limpios"] = df_reviews["tokens"].apply(limpiar_tokens)

df_reviews[["review", "tokens", "tokens_limpios"]].head(10)

,review,tokens,tokens_limpios
0,"...this looks like the kind of thing that would have warranted a massive cardboard box, at least three 3.5″ floppy disks and a chunky manual dense with evocative line-art and l...","[this, looks, like, the, kind, of, thing, that, would, have, warranted, a, massive, cardboard, box, at, least, three, floppy, disks, and, a, chunky, manual, dense, with, evocat...","[looks, like, kind, thing, warranted, massive, cardboard, box, floppy, disks, chunky, manual, dense, evocative, line, art, lore]"
1,These are the voyages of the good ship Rock Paper Spacegun. Our one year mission; to deliver a parcel.,"[these, are, the, voyages, of, the, good, ship, rock, paper, spacegun, our, one, year, mission, to, deliver, a, parcel]","[voyages, good, ship, rock, paper, spacegun, year, mission, deliver, parcel]"
2,"...a charming, retro-inspired experience. Pixel graphics and chill synth tunes create a solid package for a game that wants to bring back a familiar experience ... it seeks to ...","[a, charming, retro, inspired, experience, pixel, graphics, and, chill, synth, tunes, create, a, solid, package, for, a, game, that, wants, to, bring, back, a, familiar, experi...","[charming, retro, inspired, experience, pixel, graphics, chill, synth, tunes, create, solid, package, game, wants, bring, familiar, experience, seeks, expand, genre, item, mana..."
3,"Easy to learn, but maddeningly cerebrum-busting to master. I survived 13.2 seconds — even after multiple tries. But I didn’t want to stop indulging.","[easy, to, learn, but, maddeningly, cerebrum, busting, to, master, i, survived, seconds, even, after, multiple, tries, but, i, didn, t, want, to, stop, indulging]","[easy, learn, maddeningly, cerebrum, busting, master, survived, seconds, multiple, tries, didn, want, stop, indulging]"
4,This one will be great for fast-thinking puzzle fiends.,"[this, one, will, be, great, for, fast, thinking, puzzle, fiends]","[great, fast, thinking, puzzle, fiends]"
5,A game about literally throwing your head off every surface in a room until you reach the exit. Excellent!,"[a, game, about, literally, throwing, your, head, off, every, surface, in, a, room, until, you, reach, the, exit, excellent]","[game, literally, throwing, head, surface, room, reach, exit, excellent]"
6,I thought this is another trash game. Instead of it I was enthralled of Samurai Wish. Simple visuals but enjoyable and very hardcore gameplay. Stylish music is different for ea...,"[i, thought, this, is, another, trash, game, instead, of, it, i, was, enthralled, of, samurai, wish, simple, visuals, but, enjoyable, and, very, hardcore, gameplay, stylish, mu...","[thought, trash, game, instead, enthralled, samurai, wish, simple, visuals, enjoyable, hardcore, gameplay, stylish, music, different, level]"
7,"Slay hundreds of enemies with your Hattori Hanzo, make Uma Thurman jealous. Here comes a good top-down action game where you get to be a bloodthirsty samurai!","[slay, hundreds, of, enemies, with, your, hattori, hanzo, make, uma, thurman, jealous, here, comes, a, good, top, down, action, game, where, you, get, to, be, a, bloodthirsty, ...","[slay, hundreds, enemies, hattori, hanzo, make, uma, thurman, jealous, comes, good, action, game, bloodthirsty, samurai]"
8,"Samurai Wish is a classic topdown action game. Kill waves of enemies using your katana or throw some deadly shurikens. Great music selection (from ambiant to hiphop style), fas...","[samurai, wish, is, a, classic, topdown, action, game, kill, waves, of, enemies, using, your, katana, or, throw, some, deadly, shurikens, great, music, selection, from, ambiant...","[samurai, wish, classic, topdown, action, game, kill, waves, enemies, using, katana, throw, deadly, shurikens, great, music, selection, ambiant, hiphop, style, fast, fun, gamep..."
9,"Its narration in the form of comics, in a rather dark and violent universe, is very reminiscent of the universe of Sin City.","[its, narration, in, the, form, of, comics, in, 

### FRECUENCIA DE PALABRAS

¿qué términos se repiten más en las reseñas?

In [15]:
# Unimos todos los tokens limpios en una sola lista.
todos_los_tokens = [
    token
    for lista_tokens in df_reviews["tokens_limpios"]
    for token in lista_tokens
]

# Counter cuenta cuántas veces aparece cada palabra.
frecuencias = Counter(todos_los_tokens)

# Mostramos las 30 palabras más frecuentes como tabla.
top_frecuencias = pd.DataFrame(
    frecuencias.most_common(50),
    columns=["palabra", "frecuencia"]
)

top_frecuencias


,palabra,frecuencia
0,game,10574
1,games,3043
2,like,2656
3,fun,2559
4,great,1947
5,it's,1807
6,play,1789
7,experience,1766
8,time,1728
9,best,1709


### TF-IDF

TF-IDF permite asignar mayor peso a términos que son relevantes para un documento y menor peso a palabras que aparecen de forma demasiado general

Parámetros utilizados:

- `max_features=3000`: limita el vocabulario a un máximo de 3000 términos para mantener el análisis manejable.
- `min_df=3`: ignora términos que aparecen en menos de 3 reseñas.
- `max_df=0.85`: ignora términos presentes en más del 85% de las reseñas, porque suelen aportar poca capacidad de diferenciación.
- `lowercase=False`: el texto ya fue convertido previamente a minúsculas.



In [22]:
# Unimos los tokens limpios de cada reseña.
df_reviews["texto_limpio"] = df_reviews["tokens_limpios"].apply(" ".join)

# Eliminamos únicamente los casos donde, después de limpiar, no quedó ninguna palabra.
df_reviews = df_reviews[df_reviews["texto_limpio"].str.strip() != ""].copy()
df_reviews = df_reviews.reset_index(drop=True)

print("Reseñas disponibles para TF-IDF:", len(df_reviews))

df_reviews[["review", "texto_limpio"]].head()

Reseñas disponibles para TF-IDF: 27427


,review,texto_limpio
0,"...this looks like the kind of thing that would have warranted a massive cardboard box, at least three 3.5″ floppy disks and a chunky manual dense with evocative line-art and l...",looks like kind thing warranted massive cardboard box floppy disks chunky manual dense evocative line art lore
1,These are the voyages of the good ship Rock Paper Spacegun. Our one year mission; to deliver a parcel.,voyages good ship rock paper spacegun year mission deliver parcel
2,"...a charming, retro-inspired experience. Pixel graphics and chill synth tunes create a solid package for a game that wants to bring back a familiar experience ... it seeks to ...",charming retro inspired experience pixel graphics chill synth tunes create solid package game wants bring familiar experience seeks expand genre item management economy gamepla...
3,"Easy to learn, but maddeningly cerebrum-busting to master. I survived 13.2 seconds — even after multiple tries. But I didn’t want to stop indulging.",easy learn maddeningly cerebrum busting master survived seconds multiple tries didn want stop indulging
4,This one will be great for fast-thinking puzzle fiends.,great fast thinking puzzle fiends


In [17]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    min_df=3,
    max_df=0.85,
    lowercase=False
)

# fit_transform aprende el vocabulario y transforma cada reseña en un vector numérico.
matriz_tfidf = vectorizer.fit_transform(df_reviews["texto_limpio"])

# Nombres de las palabras que forman las columnas de la matriz.
terminos = vectorizer.get_feature_names_out()

print("Forma de la matriz TF-IDF:", matriz_tfidf.shape)
print("Cantidad de términos del vocabulario:", len(terminos))

Forma de la matriz TF-IDF: (27427, 3000)
Cantidad de términos del vocabulario: 3000


Calculamos el TF-IDF promedio de cada término

In [23]:
promedio_tfidf = np.asarray(matriz_tfidf.mean(axis=0)).ravel()

df_tfidf_promedio = pd.DataFrame({
    "termino": terminos,
    "tfidf_promedio": promedio_tfidf
})

df_tfidf_promedio = df_tfidf_promedio.sort_values(
    "tfidf_promedio",
    ascending=False
)

df_tfidf_promedio.head(50)

,termino,tfidf_promedio
1091,game,0.048165
1095,games,0.020618
1083,fun,0.019356
1550,like,0.017665
206,best,0.015756
1168,great,0.014832
1973,play,0.014748
1434,it,0.013982
911,experience,0.013033
2705,time,0.012911


Podemos utilizar la matriz TF-IDF para observar qué términos obtienen mayor peso dentro de una reseña concreta.

 #Devuelve los n términos con mayor TF-IDF de una reseña.

In [24]:
def terminos_distintivos(indice, n=8):
   
    fila = matriz_tfidf[indice].toarray().ravel()

    # Ordenamos los índices desde el peso más alto al más bajo.
    indices = fila.argsort()[::-1]

    resultado = []

    for i in indices:
        if fila[i] <= 0:
            break

        resultado.append((terminos[i], round(float(fila[i]), 4)))

        if len(resultado) == n:
            break

    return resultado

# Ejemplo con la primera reseña del corpus.
print("Juego:", df_reviews.loc[15, "name"])
print("Reseña:", df_reviews.loc[15, "review"])
print("Términos distintivos:", terminos_distintivos(15))


Juego: Whip! Whip!
Reseña: Played every coin-op in the annals of arcade history? Whip! Whip! offers something new but delightfully retro, mixing mechanics of Bubble Bobble and Bionic Commando.
Términos distintivos: [('mixing', 0.4267), ('delightfully', 0.3971), ('history', 0.3396), ('op', 0.33), ('retro', 0.2992), ('arcade', 0.2882), ('offers', 0.287), ('mechanics', 0.2642)]


### AGREGACION DEL ANALISIS POR JUEGO 

Hasta ahora la unidad de análisis fue la reseña individual. Sin embargo, el proyecto general trabaja principalmente a nivel de videojuego.

Por eso agrupamos las reseñas por `app_id` y `name`. Esto permitirá posteriormente relacionar la información textual con variables del dataset principal como propietarios estimados, votos, precio, género o tiempo de juego.

In [20]:
# Cantidad de reseñas individuales disponibles por juego.
resumen_juegos = (
    df_reviews.groupby(["app_id", "name"])
    .agg(
        cantidad_resenas=("review", "size"),
        promedio_palabras=("cantidad_palabras", "mean")
    )
    .reset_index()
)

resumen_juegos["promedio_palabras"] = resumen_juegos["promedio_palabras"].round(2)

print("Juegos con al menos una reseña procesada:", len(resumen_juegos))
resumen_juegos.sort_values("cantidad_resenas", ascending=False).head(20)

Juegos con al menos una reseña procesada: 12018


,app_id,name,cantidad_resenas,promedio_palabras
207,107600,Waves,10,8.70
214,112100,Avadon: The Black Fortress,6,26.00
239,204450,Call of Juarez: Gunslinger,6,16.33
63,24400,King Arthur - The Role-playing Wargame,6,22.33
829,265170,Acceleration of SUGURI X-Edition HD,6,38.50
245,205100,Dishonored,6,6.50
164,57800,Doc Clock: The Toasted Sandwich of Time,5,11.00
294,211440,Adventures of Shuggy,5,7.60
268,208400,Avernum: Escape From the Pit,5,30.60
9220,1849250,EA SPORTS™ WRC,5,9.60


### EXPORTACION FINAL

Guardamos una versión derivada del dataset preparada para NLP. El archivo original no se modifica.

La salida se guarda dentro de la carpeta `data/` del proyecto.

In [25]:
from pathlib import Path

# Creamos un identificador consecutivo cuando el corpus ya está terminado.
df_reviews = df_reviews.reset_index(drop=True)
df_reviews["review_id"] = np.arange(1, len(df_reviews) + 1)

COLUMNAS_EXPORTAR = [
    "review_id",
    "app_id",
    "name",
    "review",
    "idioma",
    "texto_limpio",
    "cantidad_palabras",
    "texto_original",
    "cantidad_citas"
]

RUTA_SALIDA = Path("../data/steam_games_reviews_nlp.csv")

df_reviews[COLUMNAS_EXPORTAR].to_csv(
    RUTA_SALIDA,
    index=False,
    encoding="utf-8"
)

print("Archivo generado:", RUTA_SALIDA)
print("Reseñas exportadas:", len(df_reviews))


Archivo generado: ..\data\steam_games_reviews_nlp.csv
Reseñas exportadas: 27427


## 18. Conclusiones preliminares

Durante la preparación de las reseñas se detectó que una gran parte de las filas originales contenía varias reseñas o citas concatenadas. 

Posteriormente se filtró para conservar únicamente reseñas en inglés.

El análisis de frecuencias permite identificar los términos que más se repiten, mientras que TF-IDF permite observar palabras con mayor capacidad de distinguir documentos.

Estos resultados todavía no responden por sí solos qué características hacen que un juego tenga éxito en Steam. 
El siguiente paso del proyecto será combinar el análisis textual con el dataset principal mediante `app_id` y estudiar si determinados términos o características textuales aparecen asociados con indicadores como propietarios estimados, votos positivos, precio, género o tiempo de juego.

### Limitaciones

- `langdetect` realiza una clasificación automática aproximada del idioma y puede equivocarse en textos cortos.
- La extracción de citas depende del formato de comillas `“ ”`; otros formatos pueden no detectarse de la misma forma.
- TF-IDF mide relevancia estadística de términos, pero no comprende por sí mismo el contexto completo, ironía o sentimiento.
- En esta etapa no se realiza análisis de sentimiento ni modelado predictivo.